# Digital Crime Investigation
## Full Pipeline — Data Cleaning → EDA → Anomaly Detection

**Author:** Baskara Kresna Juniarto  
**Project:** Transaction Fraud & Anomaly Analytics  

---
Upload 4 file CSV, lalu **Runtime → Run all**.

In [ ]:
# ── COLAB SETUP ─────────────────────────────────────────────────────────────
# Upload: transactions.csv, users.csv, devices.csv, merchants.csv
from google.colab import files as colab_files
import os
uploaded = colab_files.upload()
os.makedirs('/content/images', exist_ok=True)
print('Uploaded:', list(uploaded.keys()))

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.float_format', '{:,.0f}'.format)

DATA_DIR = Path('/content/')
IMG_DIR  = Path('/content/images/')
print('Ready. DATA_DIR:', DATA_DIR)

---
## Phase 1 — Data Cleaning & Validation

In [ ]:
txn       = pd.read_csv(DATA_DIR / 'transactions.csv', parse_dates=['timestamp'])
users     = pd.read_csv(DATA_DIR / 'users.csv', parse_dates=['registration_date'])
devices   = pd.read_csv(DATA_DIR / 'devices.csv', parse_dates=['first_seen','last_seen'])
merchants = pd.read_csv(DATA_DIR / 'merchants.csv')

print(f'Transactions : {len(txn):,} rows x {txn.shape[1]} cols')
print(f'Users        : {len(users):,} rows x {users.shape[1]} cols')
print(f'Devices      : {len(devices):,} rows x {devices.shape[1]} cols')
print(f'Merchants    : {len(merchants):,} rows x {merchants.shape[1]} cols')
txn.head(5)

In [ ]:
def null_report(df, label):
    null_cnt = df.isnull().sum()
    report = pd.DataFrame({'null_count': null_cnt, 'null_pct': (null_cnt/len(df)*100).round(2)})
    report = report[report.null_count > 0].sort_values('null_count', ascending=False)
    print(f'\n=== {label} ===')
    print(report if len(report) else 'No nulls ✓')

for df, name in [(txn,'transactions'),(users,'users'),(devices,'devices'),(merchants,'merchants')]:
    null_report(df, name)

print('\nDuplicates:')
print(f'  transaction_id: {txn.duplicated("transaction_id").sum()}')
print(f'  user_id       : {users.duplicated("user_id").sum()}')

In [ ]:
txn_clean = txn.copy()
txn_clean['refund_flag']     = txn_clean['refund_flag'].astype(int)
txn_clean['chargeback_flag'] = txn_clean['chargeback_flag'].astype(int)
txn_clean = txn_clean.sort_values(['user_id','timestamp']).reset_index(drop=True)
txn_clean['txn_date'] = txn_clean['timestamp'].dt.date
txn_clean['txn_hour'] = txn_clean['timestamp'].dt.hour
txn_clean['txn_dow']  = txn_clean['timestamp'].dt.dayofweek

txn_clean.to_csv(DATA_DIR / 'transactions_clean.csv', index=False)
print(f'Cleaning done. Shape: {txn_clean.shape}')
print(f'Refunds: {txn_clean["refund_flag"].sum()} | Chargebacks: {txn_clean["chargeback_flag"].sum()}')

---
## Phase 2 — Exploratory Data Analysis

In [ ]:
monthly = txn_clean.set_index('timestamp').resample('ME').agg(
    txn_count=('transaction_id','count'),
    total_amount=('amount','sum'),
    unique_users=('user_id','nunique'),
).reset_index()

fig, axes = plt.subplots(2, 1, figsize=(12, 7))
axes[0].bar(monthly['timestamp'].dt.strftime('%b %Y'), monthly['txn_count'], color='steelblue', edgecolor='white')
axes[0].set_title('Monthly Transaction Count', fontweight='bold')
axes[1].bar(monthly['timestamp'].dt.strftime('%b %Y'), monthly['total_amount']/1e9, color='darkorange', edgecolor='white')
axes[1].set_title('Monthly Total Value (Rp Billion)', fontweight='bold')
for ax in axes: ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig(IMG_DIR / '02_monthly_volume.png', dpi=150, bbox_inches='tight')
plt.show()
monthly

In [ ]:
cat_stats = txn_clean.groupby('category').agg(
    txn_count=('transaction_id','count'),
    total_amount=('amount','sum'),
    refund_count=('refund_flag','sum'),
).sort_values('txn_count', ascending=False).reset_index()
cat_stats['refund_rate'] = (cat_stats['refund_count']/cat_stats['txn_count']*100).round(1)

colors = sns.color_palette('muted', len(cat_stats))
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].barh(cat_stats['category'], cat_stats['txn_count'], color=colors)
axes[0].set_title('Transaction Count by Category', fontweight='bold')
axes[1].barh(cat_stats['category'], cat_stats['refund_rate'], color=colors)
axes[1].set_title('Refund Rate by Category (%)', fontweight='bold')
for ax in axes: ax.invert_yaxis()
plt.tight_layout()
plt.savefig(IMG_DIR / '02_category_breakdown.png', dpi=150, bbox_inches='tight')
plt.show()
cat_stats

In [ ]:
hourly = txn_clean.groupby('txn_hour').agg(
    count=('transaction_id','count'),
    refund_count=('refund_flag','sum')
).reset_index()
hourly['refund_rate'] = hourly['refund_count']/hourly['count']*100

fig, ax1 = plt.subplots(figsize=(14, 5))
ax2 = ax1.twinx()
ax1.bar(hourly['txn_hour'], hourly['count'], alpha=0.7, color='steelblue', label='Txn Count')
ax2.plot(hourly['txn_hour'], hourly['refund_rate'], 'ro-', label='Refund Rate %', linewidth=2)
ax1.set_title('Hourly Pattern — Count vs Refund Rate', fontweight='bold')
ax1.set_xticks(range(0, 24))
plt.tight_layout()
plt.savefig(IMG_DIR / '02_hourly_pattern.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
user_profile = txn_clean.groupby('user_id').agg(
    txn_count=('transaction_id','count'),
    total_amount=('amount','sum'),
    unique_devices=('device_id','nunique'),
    unique_cities=('location_id','nunique'),
    refund_count=('refund_flag','sum'),
    chargeback_count=('chargeback_flag','sum'),
).reset_index()
user_profile['refund_rate'] = user_profile['refund_count']/user_profile['txn_count']*100
top20 = user_profile.nlargest(20, 'total_amount')

fig, ax = plt.subplots(figsize=(13, 6))
colors_bar = ['crimson' if r > 10 else 'steelblue' for r in top20['refund_rate']]
ax.bar(top20['user_id'], top20['total_amount']/1e6, color=colors_bar)
ax.set_title('Top 20 Users by Spend — Red = Refund Rate > 10%', fontweight='bold')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(IMG_DIR / '02_top20_users.png', dpi=150, bbox_inches='tight')
plt.show()
top20[['user_id','txn_count','total_amount','refund_rate','unique_devices','unique_cities']]

---
## Phase 3 — Anomaly Detection & Investigation

In [ ]:
# INDICATOR A — Impossible Travel
t = txn_clean.sort_values(['user_id','timestamp']).copy()
t['prev_location']  = t.groupby('user_id')['location_id'].shift(1)
t['prev_timestamp'] = t.groupby('user_id')['timestamp'].shift(1)
t['prev_txn_id']    = t.groupby('user_id')['transaction_id'].shift(1)
t['gap_min']        = (t['timestamp'] - t['prev_timestamp']).dt.total_seconds() / 60

impossible_travel = t[
    t['prev_location'].notna() &
    (t['location_id'] != t['prev_location']) &
    (t['gap_min'] < 60)
][['user_id','prev_txn_id','transaction_id','prev_location','location_id',
   'prev_timestamp','timestamp','gap_min','amount','device_id']].copy()
impossible_travel.columns = ['user_id','from_txn','to_txn','from_city','to_city',
                              'from_time','to_time','gap_min','amount','device_id']
print(f'Impossible Travel events: {len(impossible_travel)}')
impossible_travel

In [ ]:
# INDICATOR B — Transaction Burst
def detect_burst(group, window_min=10, threshold=5):
    group = group.sort_values('timestamp').reset_index(drop=True)
    rows = []
    for i, row in group.iterrows():
        cnt = ((group['timestamp'] >= row['timestamp'] - pd.Timedelta(minutes=window_min)) &
               (group['timestamp'] <= row['timestamp'])).sum()
        if cnt >= threshold:
            rows.append({'user_id': row['user_id'], 'window_center': row['timestamp'], 'txn_in_window': cnt})
    return pd.DataFrame(rows)

burst_results = []
for uid, grp in txn_clean.groupby('user_id'):
    res = detect_burst(grp)
    if len(res): burst_results.append(res)

if burst_results:
    burst_df = pd.concat(burst_results).drop_duplicates()
    print(f'Burst users: {burst_df["user_id"].nunique()}')
    print(burst_df.groupby('user_id')['txn_in_window'].max().sort_values(ascending=False))
else:
    burst_df = pd.DataFrame()
    print('No burst detected.')

In [ ]:
# INDICATOR C — Refund Abuse
pop_avg_refund = txn_clean['refund_flag'].mean()
user_refund = txn_clean.groupby('user_id').agg(
    txn_count=('transaction_id','count'),
    refund_count=('refund_flag','sum'),
).reset_index()
user_refund['refund_rate'] = user_refund['refund_count'] / user_refund['txn_count']
refund_abusers = user_refund[
    (user_refund['txn_count'] >= 5) & (user_refund['refund_rate'] >= pop_avg_refund * 5)
].sort_values('refund_rate', ascending=False)
print(f'Pop avg refund: {pop_avg_refund*100:.2f}% | Abusers: {len(refund_abusers)}')
refund_abusers

In [ ]:
# INDICATOR D — Unusual Amount
user_cat_avg = txn_clean.groupby(['user_id','category'])['amount'].agg(avg_amt='mean', cnt='count').reset_index()
txn_with_avg = txn_clean.merge(user_cat_avg, on=['user_id','category'], how='left')
txn_with_avg['multiple'] = txn_with_avg['amount'] / txn_with_avg['avg_amt']
unusual_amount = txn_with_avg[
    (txn_with_avg['cnt'] >= 3) & (txn_with_avg['multiple'] >= 8)
][['transaction_id','user_id','timestamp','amount','category','avg_amt','multiple']].sort_values('multiple', ascending=False)
print(f'Unusual amount txn: {len(unusual_amount)} | Users: {unusual_amount["user_id"].nunique()}')
unusual_amount

In [ ]:
# INDICATOR E — New Device + High Value
p95 = txn_clean['amount'].quantile(0.95)
device_first = txn_clean.groupby(['user_id','device_id'])['timestamp'].min().reset_index()
device_first.columns = ['user_id','device_id','first_used_at']
txn_first = txn_clean.merge(device_first, on=['user_id','device_id'])
new_device_high_val = txn_first[
    (txn_first['timestamp'] == txn_first['first_used_at']) & (txn_first['amount'] > p95)
][['transaction_id','user_id','device_id','timestamp','amount','category']].sort_values('amount', ascending=False)
print(f'p95={p95:,.0f} | New device+high-value: {len(new_device_high_val)} events')
new_device_high_val

In [ ]:
# INDICATOR F — Shared Device
device_users = txn_clean.groupby('device_id').agg(
    user_count=('user_id','nunique'),
    users=('user_id', lambda x: ', '.join(sorted(x.unique()))),
    txn_count=('transaction_id','count'),
).reset_index()
shared_devices = device_users[device_users['user_count'] >= 2].sort_values('user_count', ascending=False)
print(f'Shared devices: {len(shared_devices)}')
shared_devices

In [ ]:
# INDICATOR G — Unusual Location
user_p75 = txn_clean.groupby('user_id')['amount'].quantile(0.75).reset_index()
user_p75.columns = ['user_id','p75_amt']
txn_geo = txn_clean.merge(users[['user_id','city']], on='user_id').merge(user_p75, on='user_id')
unusual_location = txn_geo[
    (txn_geo['location_id'] != txn_geo['city']) & (txn_geo['amount'] > txn_geo['p75_amt'])
][['transaction_id','user_id','city','location_id','timestamp','amount','p75_amt']].copy()
unusual_location['multiple'] = (unusual_location['amount']/unusual_location['p75_amt']).round(1)
print(f'Unusual location: {len(unusual_location)} events | {unusual_location["user_id"].nunique()} users')
unusual_location.head(10)

In [ ]:
# COMPOSITE RISK SCORE
WEIGHTS = {
    'impossible_travel': 35, 'transaction_burst': 25, 'refund_abuse': 20,
    'unusual_amount': 20, 'new_device_high_value': 20, 'shared_device': 15, 'unusual_location': 10,
}

score_map = {u: {} for u in txn_clean['user_id'].unique()}

for uid in impossible_travel['user_id'].unique():
    score_map[uid]['impossible_travel'] = WEIGHTS['impossible_travel']
if len(burst_df):
    for uid in burst_df['user_id'].unique():
        score_map[uid]['transaction_burst'] = WEIGHTS['transaction_burst']
for uid in refund_abusers['user_id'].unique():
    score_map[uid]['refund_abuse'] = WEIGHTS['refund_abuse']
for uid in unusual_amount['user_id'].unique():
    score_map[uid]['unusual_amount'] = WEIGHTS['unusual_amount']
for uid in new_device_high_val['user_id'].unique():
    score_map[uid]['new_device_high_value'] = WEIGHTS['new_device_high_value']
shared_users = [u for users_str in shared_devices['users'].str.split(', ') for u in users_str]
for uid in set(shared_users):
    if uid in score_map: score_map[uid]['shared_device'] = WEIGHTS['shared_device']
for uid in unusual_location['user_id'].unique():
    score_map[uid]['unusual_location'] = WEIGHTS['unusual_location']

risk_rows = []
for uid, indicators in score_map.items():
    total = sum(indicators.values())
    if total > 0:
        risk_rows.append({'user_id': uid, 'composite_score': total,
                          'indicators': ' | '.join(sorted(indicators.keys(), key=lambda k: -WEIGHTS[k])),
                          'indicator_count': len(indicators)})

risk_df = pd.DataFrame(risk_rows).sort_values('composite_score', ascending=False)
risk_df['risk_level'] = pd.cut(risk_df['composite_score'], bins=[0,30,60,9999], labels=['LOW','MEDIUM','HIGH'])
print(f'Flagged users: {len(risk_df)}')
print(risk_df['risk_level'].value_counts())
risk_df.head(20)

In [ ]:
level_colors = {'HIGH': 'crimson', 'MEDIUM': 'darkorange', 'LOW': 'steelblue'}
level_counts = risk_df['risk_level'].value_counts().reindex(['HIGH','MEDIUM','LOW'])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
bars = axes[0].bar(level_counts.index, level_counts.values,
                   color=[level_colors[l] for l in level_counts.index])
for bar, val in zip(bars, level_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                 str(val), ha='center', va='bottom', fontweight='bold')
axes[0].set_title('Flagged Users by Risk Level', fontweight='bold')

color_list = risk_df['risk_level'].map(level_colors)
axes[1].scatter(risk_df['indicator_count'], risk_df['composite_score'],
                c=color_list, s=80, alpha=0.7, edgecolors='white')
axes[1].axhline(y=61, color='crimson', linestyle='--', alpha=0.5, label='HIGH')
axes[1].axhline(y=31, color='darkorange', linestyle='--', alpha=0.5, label='MEDIUM')
axes[1].set_title('Risk Score vs Indicator Count', fontweight='bold')
axes[1].legend()
plt.tight_layout()
plt.savefig(IMG_DIR / '03_risk_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# CASE-001 — U0023 Impossible Travel Evidence Chain
case_u0023 = txn_clean[txn_clean['user_id']=='U0023'].sort_values('timestamp')\
    .merge(merchants[['merchant_id','merchant_name']], on='merchant_id')\
    .merge(devices[['device_id','device_type']], on='device_id')
case_u0023['gap_min'] = case_u0023['timestamp'].diff().dt.total_seconds() / 60
print('=== CASE-001: U0023 — Impossible Travel ===')
print(case_u0023[['transaction_id','timestamp','location_id','amount','device_id','device_type','gap_min']].to_string(index=False))

In [ ]:
# CASE-004 — U0045 Transaction Burst
burst_window = txn_clean[
    (txn_clean['user_id']=='U0045') &
    (txn_clean['timestamp'] >= '2024-02-14 01:00:00') &
    (txn_clean['timestamp'] <= '2024-02-14 01:10:00')
].sort_values('timestamp')

fig, ax = plt.subplots(figsize=(10, 4))
ax.scatter(burst_window['timestamp'], burst_window['amount'], s=200, color='crimson', zorder=5)
ax.plot(burst_window['timestamp'], burst_window['amount'], color='crimson', linewidth=1.5, linestyle='--', alpha=0.6)
ax.set_title('U0045 — Transaction Burst (6 × Rp500K in 6 minutes)', fontweight='bold')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'Rp{x/1e3:.0f}K'))
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig(IMG_DIR / '03_case004_burst.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# EXPORT ALL OUTPUTS
risk_df.to_csv(DATA_DIR / 'risk_scores.csv', index=False)
print('risk_scores.csv exported ✓')
print('transactions_clean.csv exported ✓')
print(f'\nImages saved: {list(IMG_DIR.glob("*.png"))}')

print('\n=== CASE REGISTER ===')
case_register = pd.DataFrame([
    {'Case':'CASE-001','Type':'Impossible Travel','Users':'U0023','Risk':'HIGH'},
    {'Case':'CASE-002','Type':'Impossible Travel','Users':'U0067','Risk':'HIGH'},
    {'Case':'CASE-003','Type':'Impossible Travel','Users':'U0112','Risk':'HIGH'},
    {'Case':'CASE-004','Type':'Transaction Burst','Users':'U0045, U0089','Risk':'HIGH'},
    {'Case':'CASE-005','Type':'Refund Abuse','Users':'U0034, U0078, U0156','Risk':'HIGH'},
    {'Case':'CASE-006','Type':'Unusual Amount','Users':'U0019, U0057, U0103, U0144, U0191','Risk':'MEDIUM'},
    {'Case':'CASE-007','Type':'New Device High Value','Users':'U0033, U0071, U0118, U0162','Risk':'MEDIUM'},
    {'Case':'CASE-008','Type':'Shared Device','Users':'D0150 (4 accts), D0151 (5 accts)','Risk':'MEDIUM'},
])
print(case_register.to_string(index=False))

In [ ]:
# DOWNLOAD SEMUA OUTPUT
import shutil
from google.colab import files as colab_files

colab_files.download('/content/transactions_clean.csv')
colab_files.download('/content/risk_scores.csv')

shutil.make_archive('/content/images_output', 'zip', '/content/images')
colab_files.download('/content/images_output.zip')
print('Download selesai!')